# 🤖 Day 144 — Streamlit: ML Model Deployment
**Month 8 | Streamlit + Python Deployment**

| Field | Detail |
|---|---|
| **Day** | 144 |
| **Topic** | `joblib` model persistence · Sidebar input widgets · Real-time prediction · Feature importance · Batch CSV prediction |
| **Dataset** | FreelanceHub India — 500 rows, seed=141 |
| **Max Score** | 80 pts + 10★ Bonus |
| **Deliverable** | `day144_app.py` — a live ML prediction tool any client can run |

---
> **Why this matters:** Month 7 taught you to *train* models. Day 144 teaches you to *deploy* them.  
> A trained `.pkl` model + Streamlit UI is a complete client product. This pattern — sidebar inputs → real-time prediction → probability display — is used in 80% of Upwork ML app gigs.

---

### Architecture you're building

```
FreelanceHub CSV
      │
      ▼
 Data Prep + Train (Random Forest, class_weight='balanced')
      │
      ▼
  Save model → freelancehub_model.pkl
      │
      ▼
  Streamlit app loads model once (st.cache_resource)
      │
      ├─► Sidebar inputs → single prediction + probability gauge
      ├─► Feature importance bar chart
      ├─► Model metrics tab
      └─► Batch CSV upload → predictions → download
```

---


---
## 🔒 Section 1 — Raw Data + Model Training
**Do not modify. Run once to generate the dataset and trained model file.**


In [1]:
# ─────────────────────────────────────────────────────────────────────────────
# SECTION 1 — Raw Data + Model Training
# Run this ONCE. Saves freelancehub_india.csv + freelancehub_model.pkl
# Do NOT modify this section.
# ─────────────────────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import joblib
import warnings
warnings.filterwarnings('ignore')

# ── Seed + dimensions
np.random.seed(141)
n = 500

# ── Categories
categories      = ['Web Development', 'Data Analytics', 'ML/AI', 'Design',
                   'Content Writing', 'Digital Marketing']
cities          = ['Mumbai', 'Delhi', 'Bangalore', 'Hyderabad', 'Chennai', 'Pune', 'Kolkata']
freelancer_lvls = ['Rising Talent', 'Level 1', 'Level 2', 'Top Rated']
statuses        = ['Completed', 'In Progress', 'Cancelled']
months          = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
                   'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']

# ── Generate raw DataFrame
df = pd.DataFrame({
    'project_id'      : [f'P{str(i).zfill(3)}' for i in range(1, n+1)],
    'category'        : np.random.choice(categories, n, p=[0.25, 0.20, 0.15, 0.15, 0.15, 0.10]),
    'budget_inr'      : np.random.randint(5000, 150001, n),
    'duration_days'   : np.random.randint(3, 91, n),
    'client_rating'   : np.round(np.random.uniform(2.5, 5.0, n), 1),
    'freelancer_level': np.random.choice(freelancer_lvls, n, p=[0.30, 0.30, 0.25, 0.15]),
    'status'          : np.random.choice(statuses, n, p=[0.65, 0.25, 0.10]),
    'city'            : np.random.choice(cities, n),
    'platform_fee_pct': np.random.choice([10, 15, 20], n, p=[0.20, 0.50, 0.30]),
    'month'           : np.random.choice(months, n)
})

# ── Save raw CSV (unchanged — never modify)
df.to_csv('freelancehub_india.csv', index=False)
print(f"✅ CSV saved: {len(df)} rows × {df.shape[1]} columns")

# ── Filter for binary classification: Completed vs Cancelled
df_clf = df[df['status'].isin(['Completed', 'Cancelled'])].copy()
df_clf['target'] = (df_clf['status'] == 'Completed').astype(int)
print(f"\nClass balance: {df_clf['target'].value_counts().to_dict()}")  # 316 vs 42

# ── Label encode categoricals
cat_cols = ['category', 'freelancer_level', 'city', 'month']
le_dict  = {}
for col in cat_cols:
    le = LabelEncoder()
    df_clf[col + '_enc'] = le.fit_transform(df_clf[col])
    le_dict[col] = le

# ── Feature list (order matters — same order must be used in app.py)
FEATURE_COLS = ['budget_inr', 'duration_days', 'client_rating',
                'platform_fee_pct',
                'category_enc', 'freelancer_level_enc', 'city_enc', 'month_enc']

X = df_clf[FEATURE_COLS]
y = df_clf['target']

# ── Train / test split (stratified — preserves 88/12 class ratio)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=141, stratify=y
)

# ── Random Forest with class_weight='balanced' to compensate for imbalance
rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=8,
    class_weight='balanced',   # ← Day 135 lesson applied here
    random_state=141
)
rf.fit(X_train, y_train)

# ── Evaluate
y_pred = rf.predict(X_test)
acc    = accuracy_score(y_test, y_pred)
print(f"\nTest Accuracy : {acc:.4f}")          # 0.8889
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Cancelled', 'Completed']))

# ── Save model artifact + encoders as a single bundle
model_bundle = {
    'model'        : rf,
    'le_dict'      : le_dict,
    'feature_cols' : FEATURE_COLS,
    'cat_cols'     : cat_cols,
    'classes'      : {col: list(le_dict[col].classes_) for col in cat_cols}
}
joblib.dump(model_bundle, 'freelancehub_model.pkl')
print("\n✅ Model bundle saved: freelancehub_model.pkl")
print("   Keys:", list(model_bundle.keys()))


✅ CSV saved: 500 rows × 10 columns

Class balance: {1: 316, 0: 42}

Test Accuracy : 0.8889

Classification Report:
              precision    recall  f1-score   support

   Cancelled       0.00      0.00      0.00         8
   Completed       0.89      1.00      0.94        64

    accuracy                           0.89        72
   macro avg       0.44      0.50      0.47        72
weighted avg       0.79      0.89      0.84        72


✅ Model bundle saved: freelancehub_model.pkl
   Keys: ['model', 'le_dict', 'feature_cols', 'cat_cols', 'classes']


---
## 📖 Section 2 — Concept Notes

### Why `joblib` for model persistence?

| Tool | Use case | Notes |
|---|---|---|
| `pickle` | General Python objects | Works but slow for large numpy arrays |
| `joblib` | sklearn models + numpy arrays | 3–10× faster, standard in production |
| `torch.save` | PyTorch models | Framework-specific |

**Rule:** Always use `joblib.dump` / `joblib.load` for sklearn models in production.

---

### `@st.cache_resource` vs `@st.cache_data`

| Decorator | What it caches | Shared across sessions? | Use for |
|---|---|---|---|
| `@st.cache_data` | DataFrames, lists, dicts | No — each user gets own copy | Loaded CSVs, computed stats |
| `@st.cache_resource` | Global objects | Yes — ONE instance for all users | ML models, DB connections |

**Critical rule:** Use `@st.cache_resource` for your model. If you use `@st.cache_data`, Streamlit tries to serialise (copy) the model for each session — slow and memory-wasteful.

---

### Sidebar inputs → prediction pipeline

```python
# 1. Collect inputs from sidebar
budget = st.sidebar.slider("Budget (₹)", 5000, 150000, 50000)

# 2. Encode categoricals using saved LabelEncoder
cat_enc = le_dict['category'].transform([selected_category])[0]

# 3. Build input DataFrame in the EXACT same column order as training
input_df = pd.DataFrame([[budget, duration, rating, fee,
                          cat_enc, lvl_enc, city_enc, month_enc]],
                        columns=FEATURE_COLS)

# 4. Predict
pred       = model.predict(input_df)[0]           # 0 or 1
prob       = model.predict_proba(input_df)[0]     # [P(Cancelled), P(Completed)]
```

**The most common bug:** Column order mismatch between training and inference. The model was trained on `FEATURE_COLS` in a specific order. If your inference DataFrame has columns in a different order, sklearn silently uses wrong features.

---

### Probability display options

```python
# Progress bar approach
st.metric("Completion Probability", f"{prob[1]*100:.1f}%")
st.progress(float(prob[1]))

# Plotly gauge (more visual)
import plotly.graph_objects as go
fig = go.Figure(go.Indicator(
    mode="gauge+number",
    value=prob[1]*100,
    gauge={'axis': {'range': [0, 100]},
           'bar': {'color': '#1F3864'}}
))
```

---

### Batch prediction pattern

```python
# User uploads new CSV → encode → predict → add column → download
uploaded = st.file_uploader("Upload project CSV for batch predictions")
if uploaded:
    df_new = pd.read_csv(io.BytesIO(uploaded.read()))   # Day 143 pattern
    # encode + predict (same pipeline as single prediction)
    df_new['predicted_status'] = model.predict(X_new)
    # download
    csv_bytes = df_new.to_csv(index=False).encode('utf-8')
    st.download_button("Download Predictions", csv_bytes, "predictions.csv")
```


---
## ✏️ Section 3 — Practice Tasks

Build `day144_app.py` task by task. Run `streamlit run day144_app.py` to verify each task live.

| Task | Topic | Points | Key pattern |
|---|---|---|---|
| T1 | Load model with `@st.cache_resource` | 10 | `joblib.load` · never reload on rerun |
| T2 | Sidebar input widgets | 15 | `selectbox` + `slider` + `number_input` → all 8 features |
| T3 | Single prediction + probability display | 15 | `predict_proba` · progress bar · colour-coded result |
| T4 | Feature importance bar chart | 15 | `px.bar` horizontal · sorted · `#1F3864` |
| T5 | Model metrics tab | 10 | Accuracy + classification report table |
| T6 | Batch CSV prediction + download | 10 | `file_uploader` · encode · predict · `download_button` |
| T7 | NRA insight (markdown cell) | 5 | Cite accuracy + class imbalance finding |
| ★  | Plotly gauge chart (Bonus) | 10★ | `go.Indicator` · mode="gauge+number" |

**Total: 80 pts + 10★ bonus**

---


### T1 — Load Model with `@st.cache_resource` (10 pts)

In [ ]:
# T1 — Model loading (put this near the top of day144_app.py)
#
# TASK: Write the load_model() function using st.cache_resource.
#       The function must:
#         1. Use @st.cache_resource decorator
#         2. Load 'freelancehub_model.pkl' with joblib
#         3. Return the full model_bundle dict
#         4. Be called ONCE at app startup — not inside any button callback
#
# WHY THIS MATTERS:
#   @st.cache_resource stores the model in memory across all reruns.
#   Without it, the 100-tree Random Forest reloads on EVERY interaction —
#   making the app slow and unusable in production.

@st.cache_resource
def load_model():
    """Load the trained model, label encoders, and feature columns."""
    try:
        bundle = joblib.load("freelancehub_model.pkl")
        return bundle
    except FileNotFoundError:
        st.error("❌ Model file 'freelancehub_model.pkl' not found. Please run the training script first.")
        st.stop()

bundle = load_model()
model = bundle["model"]
le_dict = bundle["le_dict"]
feature_cols = bundle["feature_cols"]
cat_cols = bundle["cat_cols"]
classes = bundle["classes"]



### T2 — Sidebar Input Widgets (15 pts)

In [ ]:
# T2 — Sidebar input widgets
#
# TASK: In your app's sidebar, add widgets for ALL 8 features:
#
#   budget_inr        → st.sidebar.slider("Budget (₹)", 5000, 150000, 50000, step=1000)
#   duration_days     → st.sidebar.slider("Duration (days)", 3, 90, 30)
#   client_rating     → st.sidebar.slider("Client Rating", 2.5, 5.0, 4.0, step=0.1)
#   platform_fee_pct  → st.sidebar.selectbox("Platform Fee %", [10, 15, 20])
#   category          → st.sidebar.selectbox("Category", bundle['classes']['category'])
#   freelancer_level  → st.sidebar.selectbox("Freelancer Level",
#                                             bundle['classes']['freelancer_level'])
#   city              → st.sidebar.selectbox("City", bundle['classes']['city'])
#   month             → st.sidebar.selectbox("Month", bundle['classes']['month'])
#
# ENCODING STEP (mandatory — happens after widget reads):
#   Use bundle['le_dict'][col].transform([value])[0] to convert
#   each categorical string to its integer code.
#
# COLUMN ORDER IS CRITICAL:
#   Assemble the input_df with columns in EXACTLY this order:
#   ['budget_inr', 'duration_days', 'client_rating', 'platform_fee_pct',
#    'category_enc', 'freelancer_level_enc', 'city_enc', 'month_enc']
st.sidebar.header("📝 Project Details")

budget = st.sidebar.slider("Budget (₹)", min_value=5000, max_value=150000, value=50000, step=1000)
duration = st.sidebar.slider("Duration (days)", min_value=3, max_value=90, value=30)
client_rating = st.sidebar.slider("Client Rating", min_value=2.5, max_value=5.0, value=4.0, step=0.1)
platform_fee = st.sidebar.selectbox("Platform Fee (%)", options=[10, 15, 20], index=1)

category = st.sidebar.selectbox("Category", options=classes["category"])
freelancer_level = st.sidebar.selectbox("Freelancer Level", options=classes["freelancer_level"])
city = st.sidebar.selectbox("City", options=classes["city"])
month = st.sidebar.selectbox("Month", options=classes["month"])

# Encode categoricals using saved label encoders
category_enc = le_dict["category"].transform([category])[0]
level_enc = le_dict["freelancer_level"].transform([freelancer_level])[0]
city_enc = le_dict["city"].transform([city])[0]
month_enc = le_dict["month"].transform([month])[0]

# Build input DataFrame with EXACT column order as training
input_data = [[
    budget, duration, client_rating, platform_fee,
    category_enc, level_enc, city_enc, month_enc
]]
input_df = pd.DataFrame(input_data, columns=feature_cols)


### T3 — Single Prediction + Probability Display (15 pts)

In [ ]:
# T3 — Real-time prediction
#
# TASK: Using input_df from T2, produce and display:
#   a) Prediction label ("✅ Likely COMPLETED" in green, or "⚠️ Risk of CANCELLATION" in red)
#   b) Probability numbers for both classes (round to 1 decimal %)
#   c) st.progress(float(prob[1])) — progress bar scaled to P(Completed)
#   d) A st.metric showing "Completion Probability" with the % value
#
# COLOUR LOGIC (use st.success / st.error):
#   prob[1] >= 0.60  →  st.success("✅ Likely COMPLETED")
#   prob[1] < 0.60   →  st.error("⚠️ Risk of CANCELLATION")
#
# NOTE: predict_proba returns [P(class=0), P(class=1)]
#       class=0 = Cancelled, class=1 = Completed
#       → index [0] = P(Cancelled), index [1] = P(Completed)

# Bonus: Plotly gauge chart (10★)
prob = model.predict_proba(input_df)[0]   # [P(Cancelled), P(Completed)]
pred_class = model.predict(input_df)[0]

prob_cancelled = prob[0]
prob_completed = prob[1]

# Gauge chart
fig_gauge = go.Figure(go.Indicator(
    mode="gauge+number+delta",
    value=prob_completed * 100,
    title={"text": "Completion Probability (%)"},
    delta={"reference": 50},
    gauge={
        "axis": {"range": [0, 100]},
        "bar": {"color": "#1F3864"},
        "steps": [
            {"range": [0, 40], "color": "#FCE4D6"},
            {"range": [40, 70], "color": "#FFF2CC"},
            {"range": [70, 100], "color": "#E2EFDA"},
        ],
        "threshold": {
            "line": {"color": "red", "width": 4},
            "thickness": 0.75,
            "value": 60
        }
    }
))
fig_gauge.update_layout(height=300)



### T4 — Feature Importance Bar Chart (15 pts)

In [ ]:
# T4 — Feature importance bar chart
#
# TASK: Create a horizontal Plotly bar chart of model feature importances.
#
# Requirements:
#   1. Extract importances from bundle['model'].feature_importances_
#   2. Pair with bundle['feature_cols'] in a DataFrame
#   3. Sort by importance DESCENDING
#   4. Create px.bar with:
#        x='importance', y='feature', orientation='h'
#        color_discrete_sequence=['#1F3864']
#        title='Feature Importances — What Drives Project Completion?'
#   5. Display with st.plotly_chart(fig, use_container_width=True)
#
# EXPECTED ORDER (top → bottom after sort):
#   platform_fee_pct  0.2437
#   category_enc      0.1735
#   duration_days     0.1318
#   budget_inr        0.1243
#   client_rating     0.1011
#   freelancer_level_enc 0.0941
#   city_enc          0.0737
#   month_enc         0.0579

 st.subheader("🔑 Feature Importances")
    importances = model.feature_importances_
    imp_df = pd.DataFrame({"Feature": feature_cols, "Importance": importances})
    imp_df = imp_df.sort_values("Importance", ascending=True)
    fig_imp = px.bar(
        imp_df,
        x="Importance",
        y="Feature",
        orientation="h",
        color_discrete_sequence=["#1F3864"],
        title="What Drives Project Completion? (Higher = More Influence)"
    )
    fig_imp.update_layout(yaxis=dict(categoryorder="total ascending"))
    st.plotly_chart(fig_imp, use_container_width=True)


### T5 — Model Metrics Tab (10 pts)

In [ ]:
# T5 — Model metrics in a separate tab
#
# TASK: Add a second tab ("📊 Model Metrics") alongside the prediction tab.
#       Inside it, display:
#
#   a) st.metric("Test Accuracy", "88.89%")
#   b) A table showing the classification report:
#
#      | Class     | Precision | Recall | F1-Score | Support |
#      |-----------|-----------|--------|----------|---------|
#      | Cancelled |   0.00    |  0.00  |   0.00   |    8    |
#      | Completed |   0.89    |  1.00  |   0.94   |   64    |
#
#   c) An st.warning explaining the class imbalance insight:
#      "The model achieves 88.89% accuracy but never predicts Cancellation
#       (recall = 0.00). This is because only 42/358 projects are Cancelled
#       — the model learns to always predict Completed. In a real deployment,
#       apply SMOTE or lower the prediction threshold to flag at-risk projects."
#
# STRUCTURE:
#   tab1, tab2 = st.tabs(["🔮 Predict", "📊 Model Metrics"])
#   with tab1:  <prediction UI from T3>
#   with tab2:  <metrics from this task>

tab1, tab2, tab3 = st.tabs(["🔮 Predict", "📊 Model Metrics", "📁 Batch Predict"])

with tab1:
    st.subheader("🔍 Prediction Result")
    col_left, col_right = st.columns([2, 1])
    with col_left:
        st.plotly_chart(fig_gauge, use_container_width=True)
        st.metric("Completion Probability", f"{prob_completed:.1%}")
    with col_right:
        if prob_completed >= 0.60:
            st.success("✅ Likely COMPLETED")
        else:
            st.error("⚠️ Risk of CANCELLATION")
        st.caption(f"P(Cancelled) = {prob_cancelled:.1%}")
with tab2:
    st.subheader("📈 Model Performance Metrics")
    st.metric("Test Accuracy", "88.89%")
    
    # Classification report table
    report_data = {
        "Class": ["Cancelled", "Completed"],
        "Precision": ["0.00", "0.89"],
        "Recall": ["0.00", "1.00"],
        "F1-Score": ["0.00", "0.94"],
        "Support": ["8", "64"]
    }
    st.table(pd.DataFrame(report_data))
    
    st.warning(
        "⚠️ **Class imbalance insight:** The model achieves 88.89% accuracy but never predicts Cancellation "
        "(recall = 0.00). This is because only 42 out of 358 projects are Cancelled (11.7%). The model learns "
        "to always predict Completed. In a real deployment, apply SMOTE or lower the prediction threshold "
        "to flag at-risk projects."
    )


### T6 — Batch CSV Prediction + Download (10 pts)

In [ ]:
# T6 — Batch prediction
#
# TASK: In a third tab ("📂 Batch Predict"), allow users to upload a CSV
#       and receive predictions for all rows.
#
# The uploaded CSV will have these columns:
#   category, budget_inr, duration_days, client_rating,
#   freelancer_level, city, platform_fee_pct, month
#   (same structure as freelancehub_india.csv, but WITHOUT 'status')
#
# Steps:
#   1. st.file_uploader("Upload project CSV", type=['csv'])
#   2. Read with pd.read_csv(io.BytesIO(uploaded.read()))
#   3. Encode all 4 categorical columns using bundle['le_dict']
#   4. Select FEATURE_COLS in correct order
#   5. rf.predict() + rf.predict_proba()[:, 1]
#   6. Add columns: 'predicted_status' (Completed/Cancelled) +
#                   'completion_probability' (rounded to 2dp)
#   7. st.dataframe(df_result)
#   8. st.download_button("⬇ Download Predictions",
#                          df_result.to_csv(index=False).encode('utf-8'),
#                          "batch_predictions.csv", "text/csv")
#
# HINT for encoding step:
#   for col in ['category', 'freelancer_level', 'city', 'month']:
#       df_new[col + '_enc'] = bundle['le_dict'][col].transform(df_new[col])

with tab3:
    st.subheader("📂 Batch Predictions on Multiple Projects")
    st.markdown("Upload a CSV file with columns: `category`, `budget_inr`, `duration_days`, `client_rating`, `freelancer_level`, `city`, `platform_fee_pct`, `month`")
    
    uploaded_file = st.file_uploader("Upload project CSV", type=["csv"])
    
    if uploaded_file is not None:
        try:
            df_batch = pd.read_csv(io.BytesIO(uploaded_file.read()))
            required_cols = ['category', 'budget_inr', 'duration_days', 'client_rating',
                             'freelancer_level', 'city', 'platform_fee_pct', 'month']
            missing = [c for c in required_cols if c not in df_batch.columns]
            if missing:
                st.error(f"Missing columns: {missing}")
            else:
                # Encode categoricals
                for col in cat_cols:
                    df_batch[col + '_enc'] = le_dict[col].transform(df_batch[col])
                X_batch = df_batch[feature_cols]
                preds = model.predict(X_batch)
                probs = model.predict_proba(X_batch)[:, 1]
                df_batch['predicted_status'] = np.where(preds == 1, "Completed", "Cancelled")
                df_batch['completion_probability'] = probs.round(4)
                st.dataframe(df_batch.head(10), use_container_width=True)
                csv_bytes = df_batch.to_csv(index=False).encode('utf-8')
                st.download_button(
                    label="⬇️ Download Predictions",
                    data=csv_bytes,
                    file_name="batch_predictions.csv",
                    mime="text/csv"
                )
        except Exception as e:
            st.error(f"Error processing file: {e}")


### T7 — NRA Insight (5 pts)

In [ ]:
# T7 — NRA Insight (write in a markdown cell or st.markdown in the app)
#
# TASK: Write a 3-sentence NRA insight that a client would find actionable.
#
# NRA FORMAT:
#   Number  — cite a specific figure from the model output
#   Reason  — explain the business mechanism behind it
#   Action  — prescribe a specific, named intervention
#
# ANCHOR POINTS TO USE:
#   - Accuracy: 88.89% overall
#   - Top feature: platform_fee_pct (importance = 0.2437)
#   - Class imbalance: only 42 of 358 projects are Cancelled (11.7%)
#   - Model never predicts Cancellation (Recall = 0.00 for Cancelled class)
#
# TEMPLATE (fill in the blanks — do NOT just copy this):
#   "The model achieves __% accuracy on the test set, driven primarily by
#    __ (importance = __), suggesting that __ is the strongest signal for
#    project completion. However, with only __% of projects being Cancelled,
#    the model __ — meaning real at-risk projects are missed entirely.
#    To fix this, [specific action: name the technique + threshold]."

st.markdown("---")
st.subheader("📌 NRA Insight: Managing Project Completion Risk")
st.markdown("""
**Number:** The model achieves **88.89% accuracy**, with `platform_fee_pct` as the strongest predictor (importance = **0.24**).  
**Reason:** Only **11.7%** of projects are Cancelled (42 out of 358). The model never predicts cancellation (recall = 0.00) – it always chooses "Completed" because that maximises accuracy.  
**Action:** Lower the prediction threshold from 0.50 to **0.30**. Flag any project with P(Completed) < 0.70 for manual review. Alternatively, retrain with **SMOTE** to synthetically balance the classes, and set a minimum fee discount for high-budget projects to reduce cancellation risk.
""")


### ★ Bonus — Plotly Gauge Chart (10 pts)

In [ ]:
# ★ BONUS — Replace st.progress with a Plotly Indicator gauge
#
# TASK: Replace the st.progress bar in T3 with a Plotly gauge chart.
#
# Requirements:
#   import plotly.graph_objects as go
#
#   fig = go.Figure(go.Indicator(
#       mode    = "gauge+number+delta",
#       value   = prob[1] * 100,             # P(Completed) as percentage
#       title   = {'text': "Completion Probability (%)"},
#       delta   = {'reference': 50},         # shows +/- vs 50% baseline
#       gauge   = {
#           'axis' : {'range': [0, 100]},
#           'bar'  : {'color': '#1F3864'},
#           'steps': [
#               {'range': [0, 40],  'color': '#FCE4D6'},   # red zone
#               {'range': [40, 70], 'color': '#FFF2CC'},   # amber zone
#               {'range': [70, 100],'color': '#E2EFDA'},   # green zone
#           ],
#           'threshold': {
#               'line' : {'color': 'red', 'width': 4},
#               'thickness': 0.75,
#               'value': 60                  # 60% decision boundary
#           }
#       }
#   ))
#   st.plotly_chart(fig, use_container_width=True)
#
# EXPECTED OUTPUT for sample (ML/AI, ₹80k, 30d, 4.2★, Level 2, Bangalore, Mar):
#   P(Completed)  = 0.8474 → gauge shows 84.7%, needle in green zone

# Write your bonus code here ↓


---
## 📊 Section 4 — Scoring Rubric

| Task | Max | Full marks criteria | Common deductions |
|---|---|---|---|
| T1 | 10 | `@st.cache_resource` used; `joblib.load`; model loads once at startup | −5 if `@st.cache_data` used instead; −3 if loaded inside button callback |
| T2 | 15 | All 8 widgets present; correct ranges; categorical encoding uses `le_dict`; columns in correct order | −3 per missing widget; −5 if encoding absent; −5 if column order wrong |
| T3 | 15 | Both probabilities shown; `st.success`/`st.error` conditional; progress bar; `st.metric` | −5 if prob indices swapped (0=Cancelled, 1=Completed); −3 if no colour logic |
| T4 | 15 | Horizontal bar chart; sorted descending; `#1F3864` colour; `use_container_width=True` | −5 if not sorted; −3 if orientation not horizontal; −2 if no title |
| T5 | 10 | Tab structure present; accuracy = 88.89%; imbalance warning included | −5 if no tab; −3 if imbalance warning missing |
| T6 | 10 | File uploader present; encoding uses saved `le_dict`; predictions added; `.encode('utf-8')` on download | −5 if fresh LabelEncoder used; −3 if `.encode('utf-8')` missing |
| T7 | 5 | Specific number cited; imbalance reason stated; named action (threshold or SMOTE) | −2 generic action; −1 no number cited |
| ★ Bonus | 10 | Gauge chart renders; 3 colour zones; threshold line at 60; delta reference = 50 | −3 if zones missing; −2 if no threshold line |

**Total: 80 pts + 10★ bonus**

---

## 🏁 Scorecard

| Day | Topic | Score |
|---|---|---|
| 141 | Streamlit Fundamentals | 80/80 + 10★ |
| 142 | Session State + Tabs + Forms | 80/80 + 10★ |
| 143 | File Upload + Dynamic EDA | — / 80 + 10★ |
| **144** | **ML Model Deployment** | **— / 80 + 10★** |

---

## 🎤 Interview / Client Answer

**Q: "How do you deploy a machine learning model as a web application?"**

> "I separate deployment into two phases. First, I train and serialize the model using `joblib` — this produces a `.pkl` file that captures both the model and all preprocessing objects like LabelEncoders. Second, I build a Streamlit app that loads this bundle once using `@st.cache_resource`, meaning the model stays in memory instead of reloading on every interaction. The app collects feature inputs via sidebar widgets, encodes categoricals using the same encoders from training to prevent any mapping mismatch, and calls `predict_proba` to show not just a binary label but a confidence percentage. For batch workflows I add a file uploader that lets clients paste their own data and download predictions as CSV. The whole thing runs as `streamlit run app.py` — no server configuration required."

---

## 💡 Key Takeaway — Day 144

**`@st.cache_resource` is the deployment pattern that makes ML apps fast.**
Without it, a 100-tree Random Forest reloads on every Streamlit rerun — making the app unusably slow. With it, the model loads once and all interactions are sub-second. Combine this with column-order discipline (training and inference must use identical feature order) and encoding via saved LabelEncoders (not freshly fit ones), and you have a production-grade deployment pattern that works identically on Streamlit Cloud or a client's private server.
